# 08.04 -- DPO: Direct Preference Optimisation

**Module:** 08 -- Alignment  
**Notebook:** 4 of 5  
**Prerequisites:** 08.02 (Reward Models), 08.03 (RLHF PPO)

---

## Learning Objectives

1. Derive the DPO loss from the RLHF objective mathematically
2. Understand exactly what DPO does that makes it equivalent to RLHF without explicit RL
3. Implement DPO training from scratch using the derived loss formula
4. Train a model with trl's DPOTrainer and read the training diagnostics
5. Understand the failure modes specific to DPO: distribution mismatch and length exploitation

---

## 1. Motivation: Why Simplify RLHF?

RLHF with PPO requires four models loaded simultaneously: the policy, the reference model (frozen), the reward model, and the value head. For a 7B model, this requires 4 * ~14 GB = ~56 GB of GPU memory at minimum, before accounting for activations and gradients.

DPO (Rafailov et al., 2023) shows that the RLHF optimisation problem has an analytical solution: the optimal policy under the RLHF objective can be expressed in closed form in terms of the reference policy and the reward. This allows us to directly optimise the policy on preference data without:
- Training a separate reward model
- Running PPO
- Maintaining a value head
- Generating rollouts during training

The result is a training procedure as simple as standard supervised fine-tuning.

---

## 2. The DPO Derivation

Starting from the RLHF objective (from Notebook 08.03):

```
J(pi) = E[r(x,y)] - beta * KL[pi || pi_ref]
```

The optimal policy that maximises this objective is:

```
pi*(y|x) = pi_ref(y|x) * exp(r(x,y) / beta) / Z(x)

where Z(x) = sum_y pi_ref(y|x) * exp(r(x,y)/beta)  is a normalising partition function
```

Rearranging to express the reward in terms of the policies:

```
r(x,y) = beta * log(pi*(y|x) / pi_ref(y|x)) + beta * log Z(x)
```

The Bradley-Terry preference probability becomes:

```
P(y_w > y_l | x) = sigmoid(r(x,y_w) - r(x,y_l))
                 = sigmoid(
                     beta * log(pi*(y_w|x) / pi_ref(y_w|x))
                   - beta * log(pi*(y_l|x) / pi_ref(y_l|x))
                   )
```

Note that Z(x) cancels! This is the key insight. The DPO loss is:

```
L_DPO = -E[log sigmoid(
    beta * (log pi(y_w|x) - log pi_ref(y_w|x))
  - beta * (log pi(y_l|x) - log pi_ref(y_l|x))
)]
```

This is a standard binary cross-entropy loss with no RL, no reward model, and no sampling.

In [ ]:
# Environment setup.
#
# DPO requires only the standard fine-tuning libraries plus trl.
# Notably, DPO does NOT require the bitsandbytes package that RLHF needs
# for quantised reward model inference, making it more lightweight.

!pip install transformers peft datasets trl accelerate matplotlib scipy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 3. Implementing the DPO Loss From Scratch

In [ ]:
# DPO loss implemented directly from the derivation above.
#
# The inputs are the per-token log-probabilities of the chosen and rejected
# responses under the policy being trained, and under the frozen reference
# policy. These are already-computed values from two forward passes.
#
# The log-probability of a full sequence under a language model is the sum
# of per-token log-probabilities:
#   log p(y|x) = sum_t log p(y_t | x, y_<t)
#
# We only sum over the response tokens, not the prompt tokens. This mirrors
# the prompt masking from Notebook 07.03: we only care about the quality of
# the response, not how well the policy reproduces the prompt.
#
# The beta parameter controls the trade-off between following preferences
# and staying close to the reference policy. A larger beta means the policy
# is penalised more for deviating from the reference -- equivalent to a
# stronger KL constraint in PPO.

def dpo_loss(
    policy_logprob_chosen:   torch.Tensor,   # (batch,) sum of log-probs for chosen response
    policy_logprob_rejected: torch.Tensor,   # (batch,) sum of log-probs for rejected response
    ref_logprob_chosen:      torch.Tensor,   # (batch,) reference model log-probs, chosen
    ref_logprob_rejected:    torch.Tensor,   # (batch,) reference model log-probs, rejected
    beta:                    float = 0.1,    # KL constraint strength
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Direct Preference Optimisation loss (Rafailov et al., 2023).

    Returns:
        loss:           scalar training loss
        chosen_reward:  implicit reward for chosen responses (for logging)
        rejected_reward: implicit reward for rejected responses (for logging)
    """
    # Implicit reward: how much more likely is this response under policy vs reference?
    # This is the 'reward' that the optimal RLHF policy would assign, re-expressed
    # in terms of policy log-probability ratios.
    chosen_reward   = beta * (policy_logprob_chosen   - ref_logprob_chosen)
    rejected_reward = beta * (policy_logprob_rejected - ref_logprob_rejected)

    # The preference probability: sigmoid of the chosen-minus-rejected reward
    # This is exactly the Bradley-Terry formula applied to the implicit rewards
    reward_diff = chosen_reward - rejected_reward

    # DPO loss: negative log-likelihood of the correct preference ordering
    loss = -F.logsigmoid(reward_diff).mean()

    return loss, chosen_reward.detach(), rejected_reward.detach()


def get_sequence_logprob(
    model,
    input_ids:      torch.Tensor,   # (1, seq_len) full sequence (prompt + response)
    attention_mask: torch.Tensor,
    response_start: int,            # token index where the response begins
) -> torch.Tensor:                  # scalar: sum of response token log-probs
    """
    Compute the sum of per-token log-probabilities for the response portion only.

    The language model output at position t is a distribution over token t+1.
    So output[i] corresponds to the probability of input_ids[i+1].
    We extract only the positions corresponding to response tokens.
    """
    with torch.no_grad() if not model.training else torch.enable_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits  # (1, seq_len, vocab_size)

    # Shift: logits[t] predicts token[t+1]
    # Response tokens start at response_start in the input_ids
    # So we need logits from (response_start-1) to (seq_len-1)
    response_logits = logits[0, response_start-1:-1, :]  # (resp_len, vocab)
    response_tokens = input_ids[0, response_start:]       # (resp_len,)

    # Gather the log-probability of each actual response token
    log_probs = F.log_softmax(response_logits, dim=-1)
    token_log_probs = log_probs.gather(1, response_tokens.unsqueeze(1)).squeeze(1)

    return token_log_probs.sum()


print("DPO loss and log-probability functions defined.")
print()
print("Key distinction from reward model training:")
print("  Reward model: r_chosen and r_rejected are outputs of a separate RM network")
print("  DPO:          'implicit rewards' are derived from the policy itself vs reference")
print("                No separate reward network is needed.")

In [ ]:
# Demonstrate the DPO loss on synthetic log-probability values.
#
# We test the loss under three scenarios that represent different stages
# of DPO training:
#
# Scenario 1: Before training. The policy is the reference model, so
#   policy_logprob == ref_logprob for both chosen and rejected.
#   Implicit rewards are zero, reward_diff is zero, loss is log(2) ~ 0.693.
#   This is the expected initialisation loss for DPO.
#
# Scenario 2: Policy prefers chosen. The policy assigns higher log-probability
#   to chosen than the reference does, and lower to rejected.
#   Implicit reward for chosen > implicit reward for rejected -> lower loss.
#
# Scenario 3: Policy prefers rejected (wrong direction).
#   Higher loss; gradient will push the policy back toward the correct ordering.

import math

scenarios = [
    {
        'label': 'Before training (policy == reference)',
        'policy_chosen':   torch.tensor([-12.0, -15.0, -10.0]),
        'policy_rejected': torch.tensor([-14.0, -13.0, -11.0]),
        'ref_chosen':      torch.tensor([-12.0, -15.0, -10.0]),  # same as policy
        'ref_rejected':    torch.tensor([-14.0, -13.0, -11.0]),
    },
    {
        'label': 'Mid-training (policy prefers chosen)',
        'policy_chosen':   torch.tensor([-10.0, -13.0, -9.0]),   # improved over reference
        'policy_rejected': torch.tensor([-16.0, -15.0, -13.0]),  # reduced over reference
        'ref_chosen':      torch.tensor([-12.0, -15.0, -10.0]),
        'ref_rejected':    torch.tensor([-14.0, -13.0, -11.0]),
    },
    {
        'label': 'Wrong direction (policy prefers rejected)',
        'policy_chosen':   torch.tensor([-15.0, -18.0, -13.0]),  # worse than reference
        'policy_rejected': torch.tensor([-12.0, -11.0, -9.0]),   # better than reference
        'ref_chosen':      torch.tensor([-12.0, -15.0, -10.0]),
        'ref_rejected':    torch.tensor([-14.0, -13.0, -11.0]),
    },
]

print(f"{'Scenario':<50} {'Loss':>8} {'Acc':>8}")
print("-" * 70)
for s in scenarios:
    loss, chosen_rew, rejected_rew = dpo_loss(
        s['policy_chosen'], s['policy_rejected'],
        s['ref_chosen'],    s['ref_rejected'],
        beta=0.1
    )
    acc = (chosen_rew > rejected_rew).float().mean().item()
    print(f"{s['label']:<50} {loss.item():>8.4f} {acc:>8.1%}")

print()
print(f"Expected initial loss (all rewards zero): {math.log(2):.4f}")
print("The DPO loss at initialisation (when policy == reference) is always log(2).")
print("This is a useful sanity check: if your initial loss is far from 0.693,")
print("something is wrong with your reference model or tokenisation.")

## 4. Full DPO Training with trl

In [ ]:
# Prepare the preference dataset in the format DPOTrainer expects.
#
# DPOTrainer requires a dataset with three columns:
#   - 'prompt':   the instruction text (without the response)
#   - 'chosen':   the preferred response
#   - 'rejected': the dispreferred response
#
# Note that 'prompt', 'chosen', and 'rejected' should NOT include the
# formatted template markers. DPOTrainer applies the tokenizer's chat template
# internally. If your model does not have a chat template, you can pass a
# custom formatting_func argument to DPOTrainer.
#
# The training data here is the same preference data used in Notebook 08.02.
# In a real project, you would use a larger, more diverse preference dataset
# such as Anthropic's HH-RLHF dataset or the UltraFeedback dataset.

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from trl import DPOTrainer, DPOConfig

PREFERENCE_PAIRS = [
    {
        'prompt': 'Explain what dropout regularisation does in neural networks.',
        'chosen': 'Dropout randomly deactivates a fraction of neurons during each training step, with probability p. This prevents co-adaptation: neurons cannot rely on specific others always being present, so each must learn independently useful features. At inference, all neurons are active and outputs are scaled by (1-p) to maintain the expected activation magnitude. Dropout is a form of ensemble learning: it approximates averaging over exponentially many sub-networks.',
        'rejected': 'Dropout is a regularisation technique that helps prevent overfitting by randomly turning off neurons.',
    },
    {
        'prompt': 'What is the difference between a validation set and a test set?',
        'chosen': 'The validation set is used during training to tune hyperparameters and make decisions like when to stop training. Because validation metrics inform model development decisions, the model is indirectly optimised for the validation set. The test set is held out completely until after all development is finished and serves as an unbiased estimate of generalisation performance. Evaluating on the test set multiple times during development constitutes test set leakage and will produce optimistically biased results.',
        'rejected': 'A validation set is for validating the model during training. A test set is for testing after training. They are both used to evaluate model performance.',
    },
    {
        'prompt': 'What causes NaN loss during neural network training?',
        'chosen': 'NaN loss typically has four causes: (1) Exploding gradients -- use gradient clipping (max_norm=1.0) and reduce the learning rate. (2) Invalid inputs -- check for NaN or infinity in your data and labels. (3) Numerical instability in loss functions -- use log_softmax instead of log(softmax(...)) and ensure you are not dividing by zero. (4) Learning rate too high -- the loss jumps to infinity on the first step. Start by reducing the learning rate by 10x and checking inputs first.',
        'rejected': 'NaN loss can happen because of various numerical problems. Try reducing the learning rate and checking your data.',
    },
    {
        'prompt': 'Explain why residual connections help train deep networks.',
        'chosen': 'Residual connections (He et al., 2016) add the input of a layer directly to its output: y = F(x) + x. This has two effects. First, gradients can flow directly through the skip connection without passing through the non-linear transformation F, which alleviates the vanishing gradient problem in very deep networks. Second, each layer needs only to learn a residual correction to the identity function rather than a full transformation from scratch, which makes initialisation easier and reduces the effective depth of the optimisation problem.',
        'rejected': 'Residual connections help because they let gradients flow more easily. They add skip connections that bypass layers.',
    },
    {
        'prompt': 'What is the purpose of temperature scaling in language model generation?',
        'chosen': 'Temperature scales the logits before the softmax: p(x) = softmax(logits / T). At T=1 (default), the distribution is unchanged. T < 1 sharpens the distribution, making the most likely tokens even more probable and producing more deterministic, repetitive output. T > 1 flattens the distribution, increasing randomness and diversity at the cost of coherence. T approaches 0 corresponds to greedy decoding. Common values for creative generation are 0.7-1.2; factual tasks use lower values around 0.1-0.5.',
        'rejected': 'Temperature controls how random the model is. Low temperature makes it more deterministic and high temperature makes it more creative.',
    },
] * 6  # replicate for demo purposes

dpo_dataset = Dataset.from_list(PREFERENCE_PAIRS)

print(f"DPO training dataset: {len(dpo_dataset)} preference pairs")
print()
print("Dataset features:", dpo_dataset.features)
print()

# Verify lengths: chosen responses should generally be longer than rejected
chosen_lens   = [len(ex['chosen'].split())   for ex in dpo_dataset]
rejected_lens = [len(ex['rejected'].split()) for ex in dpo_dataset]
print(f"Mean chosen length:   {np.mean(chosen_lens):.0f} words")
print(f"Mean rejected length: {np.mean(rejected_lens):.0f} words")
print(f"Chosen longer than rejected: {np.mean([c > r for c, r in zip(chosen_lens, rejected_lens)]):.0%} of pairs")

In [ ]:
# Load the model and tokeniser for DPO training.
#
# In DPO, the reference model is the SFT model before alignment training.
# trl's DPOTrainer automatically creates a frozen copy of the model as the
# reference at the start of training. This means you only need to load one
# model (the policy), not two -- a significant memory advantage over RLHF.
#
# We add LoRA adapters to the policy model so that only the adapter weights
# are updated during DPO. The frozen base model weights serve implicitly as
# the reference (before LoRA modifications are applied). This is an elegant
# property of LoRA-based DPO: the reference model is the base weights,
# and the policy is the base weights plus the learned LoRA deltas.

MODEL_NAME = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

policy_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['c_attn', 'c_proj'],
    task_type=TaskType.CAUSAL_LM,
    bias='none',
)
policy_model = get_peft_model(policy_model, lora_config)
policy_model.print_trainable_parameters()

print()
print("Memory advantage of LoRA-DPO over RLHF:")
print("  RLHF (PPO): policy + reference + reward model + value head = 4 models")
print("  DPO + LoRA: base weights (shared) + small LoRA deltas = effectively 1.something models")

In [ ]:
# Configure and run DPO training.
#
# The DPOConfig arguments to understand:
#
#   beta: the KL constraint coefficient. Directly corresponds to the beta
#         in the mathematical derivation. Larger beta = more conservative,
#         stays closer to reference. Typical range: 0.01 to 0.5.
#         For instruction-following: 0.1-0.2 works well.
#         For RLHF-like behaviour: 0.01-0.05.
#
#   loss_type: the DPO loss variant. Options:
#     'sigmoid': original DPO loss (Rafailov et al., 2023)
#     'hinge':   SLiC-HF (Zhao et al., 2023) -- no gradient for easy pairs
#     'ipo':     IPO (Azar et al., 2024) -- avoids degenerate solutions
#     'simpo':   SimPO (Meng et al., 2024) -- no reference model needed
#
#   max_prompt_length and max_length: control tokenisation truncation.
#     max_prompt_length limits how long the prompt can be.
#     max_length limits the full (prompt + response) length.
#     Setting these too short will truncate responses and hurt quality.
#
# We log chosen_rewards and rejected_rewards: the per-batch mean implicit
# rewards. The difference (margin) should grow during training, indicating
# that the policy is increasingly distinguishing chosen from rejected.

from transformers import TrainerCallback

class DPOLogger(TrainerCallback):
    """Records DPO-specific metrics at each logging step."""
    def __init__(self):
        self.log_history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self.log_history.append({
                'step':             state.global_step,
                'loss':             logs.get('loss', None),
                'chosen_reward':    logs.get('rewards/chosen', None),
                'rejected_reward':  logs.get('rewards/rejected', None),
                'reward_margin':    logs.get('rewards/margins', None),
                'reward_accuracy':  logs.get('rewards/accuracies', None),
            })

dpo_logger = DPOLogger()

dpo_config = DPOConfig(
    output_dir='./dpo_output',
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    beta=0.1,                    # KL constraint coefficient
    loss_type='sigmoid',         # standard DPO loss
    max_prompt_length=128,
    max_length=256,
    logging_steps=4,
    report_to='none',
    dataloader_num_workers=0,
)

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,   # None = use the frozen base model as reference (LoRA setup)
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    callbacks=[dpo_logger],
)

print("Starting DPO training...")
dpo_trainer.train()
print("DPO training complete.")

In [ ]:
# Visualise DPO training diagnostics.
#
# DPO has four key metrics to monitor during training:
#
# 1. Loss: should decrease. Initial value should be near log(2) ~ 0.693.
#    If the initial loss is much higher, the reference model log-probs are
#    computed incorrectly (wrong tokenisation or wrong model).
#
# 2. Chosen reward: the implicit reward for chosen responses.
#    Should increase as the policy learns to prefer chosen responses.
#
# 3. Rejected reward: the implicit reward for rejected responses.
#    Should decrease (or increase more slowly than chosen reward).
#
# 4. Reward accuracy: the fraction of pairs where chosen_reward > rejected_reward.
#    This is the DPO analogue of the preference accuracy metric in Notebook 08.02.
#    Should start near 50% and increase toward 80-90% on the training set.
#    Be cautious if it reaches 100%: the model may be overfit.

logs = dpo_logger.log_history
steps = [l['step'] for l in logs if l['loss'] is not None]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

def plot_metric(ax, key, label, color, note):
    values = [l[key] for l in logs if l.get(key) is not None]
    valid_steps = steps[:len(values)]
    if values:
        ax.plot(valid_steps, values, 'o-', color=color, linewidth=2, markersize=5)
    ax.set_xlabel('Training Step', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(f'{label}\n({note})', fontsize=11)
    ax.grid(alpha=0.3)

plot_metric(axes[0,0], 'loss',            'Training Loss',       '#1f77b4', 'Should decrease; start near 0.693')
plot_metric(axes[0,1], 'reward_accuracy', 'Reward Accuracy',     '#2ca02c', 'Should increase from ~50%')
plot_metric(axes[1,0], 'chosen_reward',   'Chosen Reward',       '#2ca02c', 'Should increase (policy prefers chosen more)')
plot_metric(axes[1,1], 'rejected_reward', 'Rejected Reward',     '#d62728', 'Should decrease (policy assigns less probability to rejected)')

# Add reference line for initial loss
axes[0,0].axhline(np.log(2), color='orange', linestyle='--', alpha=0.7, label='Expected initial loss (log 2)')
axes[0,0].legend(fontsize=8)

plt.suptitle('DPO Training Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_dpo_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise DPO loss variants and their properties.
#
# The original DPO paper uses the sigmoid loss. Several variants have been
# proposed that address specific failure modes:
#
# Sigmoid (DPO): -log(sigmoid(beta * (r_w - r_l)))
#   Standard DPO. Smooth gradient everywhere. Can overfit to training pairs.
#
# Hinge (SLiC): max(0, margin - (r_w - r_l))
#   Zero loss when the margin is already satisfied. Focuses gradient on
#   hard examples. Less likely to overfit.
#
# IPO: ((r_w - r_l) - 1/(2*beta))^2
#   Solves the overoptimisation problem of DPO by targeting a specific
#   margin rather than pushing the difference to infinity.
#
# All variants accept the same input (r_w - r_l) and return a scalar loss.

reward_diffs = np.linspace(-3, 4, 300)
beta  = 0.1
margin = 0.5

loss_sigmoid = -np.log(1 / (1 + np.exp(-reward_diffs)))  # DPO sigmoid
loss_hinge   = np.maximum(0, margin - reward_diffs)       # SLiC hinge
target_diff  = 1 / (2 * beta)   # IPO target
loss_ipo     = (reward_diffs - target_diff) ** 2          # IPO quadratic

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(reward_diffs, loss_sigmoid / loss_sigmoid.max(), linewidth=2.5,
        color='#1f77b4', label='DPO sigmoid (original)')
ax.plot(reward_diffs, loss_hinge   / loss_hinge.max(),   linewidth=2.5,
        color='#2ca02c', label=f'SLiC hinge (margin={margin})')
ax.plot(reward_diffs, loss_ipo     / loss_ipo.max(),     linewidth=2.5,
        color='#ff7f0e', label=f'IPO quadratic (target={target_diff:.1f})')

ax.axvline(0, color='gray', linestyle='--', alpha=0.6, label='Decision boundary')
ax.axvline(target_diff, color='orange', linestyle=':', alpha=0.5, label=f'IPO target ({target_diff:.1f})')
ax.set_xlabel('Reward Difference (chosen - rejected)', fontsize=11)
ax.set_ylabel('Normalised Loss', fontsize=11)
ax.set_title('DPO Loss Variants\n'
             'IPO penalises exceeding the target margin; DPO pushes difference to infinity', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/08_dpo_variants.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key difference between DPO and IPO:")
print("  DPO sigmoid: zero gradient as reward_diff -> infinity (can overfit)")
print("  IPO quadratic: always penalises being far from the target margin")
print("  IPO is preferred when the preference dataset is noisy or small.")

## 5. DPO Failure Modes

DPO is simpler than RLHF but has its own failure modes that every practitioner should know.

### 5.1 Distribution Mismatch
DPO is trained on static preference pairs where responses were generated by some policy (often GPT-4 or human writers). If the policy to be trained is very different from the policy that generated the training data, the preference model does not generalise well. This is the 'distribution mismatch' problem.

The fix is **iterative DPO (online DPO)**: periodically sample responses from the current policy, collect human preferences on those samples, and retrain. This keeps the training distribution close to the policy distribution.

### 5.2 Length Exploitation
Just like reward models (Notebook 08.02), DPO models can exploit length. If chosen responses are consistently longer than rejected responses in the training data (which is often true), the model learns to prefer verbosity. The fix is **SimPO**, which normalises log-probabilities by sequence length before computing the implicit reward.

In [ ]:
# Demonstrate and quantify length exploitation in DPO.
#
# We measure whether the DPO training dataset has a systematic length
# imbalance: if chosen responses are consistently longer than rejected,
# the model will associate length with quality.
#
# The SimPO (Simple Preference Optimisation) fix is to normalise the
# per-sequence log-probability by sequence length before computing the
# implicit reward. This removes the length correlation from the signal.
#
# SimPO also removes the reference model entirely, replacing the KL
# constraint with a length-normalised margin target. This further simplifies
# the training setup.

from scipy import stats as scipy_stats

chosen_lengths   = [len(p['chosen'].split())   for p in PREFERENCE_PAIRS[:30]]
rejected_lengths = [len(p['rejected'].split()) for p in PREFERENCE_PAIRS[:30]]
length_diffs     = [c - r for c, r in zip(chosen_lengths, rejected_lengths)]

# Statistical test: is chosen reliably longer than rejected?
t_stat, p_val = scipy_stats.ttest_rel(chosen_lengths, rejected_lengths)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(rejected_lengths, chosen_lengths, alpha=0.6, color='#1f77b4', s=40)
max_len = max(max(chosen_lengths), max(rejected_lengths)) + 10
ax.plot([0, max_len], [0, max_len], 'k--', alpha=0.4, label='Equal length')
ax.set_xlabel('Rejected response length (words)', fontsize=11)
ax.set_ylabel('Chosen response length (words)', fontsize=11)
ax.set_title('Length Imbalance in DPO Training Data\n'
             'Points above the diagonal: chosen is longer than rejected', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(length_diffs, bins=15, color='#2ca02c', alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', alpha=0.6, label='No difference')
ax.axvline(np.mean(length_diffs), color='red', linestyle='--',
           label=f'Mean diff = {np.mean(length_diffs):.0f} words')
ax.set_xlabel('Chosen - Rejected length (words)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'Length Difference Distribution\n'
             f't-test: t={t_stat:.2f}, p={p_val:.4f}', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('DPO Length Exploitation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_dpo_length.png', dpi=150, bbox_inches='tight')
plt.show()

pct_chosen_longer = np.mean([d > 0 for d in length_diffs])
print(f"Chosen responses longer than rejected: {pct_chosen_longer:.0%}")
print(f"Mean length difference: {np.mean(length_diffs):.1f} words")
print(f"Paired t-test p-value: {p_val:.4f}")
print()
if p_val < 0.05:
    print("Statistically significant length imbalance. The model may learn length as a quality proxy.")
    print("Consider using loss_type='simpo' or manually balancing lengths in your dataset.")
else:
    print("No significant length imbalance in this dataset.")

## 6. RLHF vs DPO: Practical Decision Guide

```
When to choose DPO over RLHF:
  - Limited GPU memory (DPO needs ~2x model memory; RLHF needs ~4x)
  - Stable, fixed preference dataset (no need for online data collection)
  - First alignment iteration (iterate quickly before committing to RLHF)
  - Quality target is 'good enough' rather than state-of-the-art

When to choose RLHF over DPO:
  - Online data collection: continuously updating preferences on current model outputs
  - Composite rewards: multiple reward signals that are hard to collapse into preference pairs
  - Production safety-critical systems: explicit reward model is easier to audit and monitor
  - Peak quality: RLHF with a strong reward model + careful PPO still leads on some benchmarks
```

## 7. Exercises

1. **Beta sweep**: Train DPO with beta values [0.01, 0.05, 0.1, 0.5]. After training, measure reward accuracy and the mean reward margin on a held-out set. Plot accuracy vs margin. Which beta achieves the best trade-off?

2. **Loss variant comparison**: Train two models -- one with `loss_type='sigmoid'` and one with `loss_type='ipo'`. Evaluate both with an LLM judge on 20 held-out prompts. Which produces better responses?

3. **Length normalisation**: Manually modify the DPO loss to normalise log-probabilities by sequence length before computing the implicit reward. Retrain and measure whether the correlation between response length and reward accuracy decreases.

4. **Reference model matters**: Train DPO starting from the base GPT-2 model (no SFT) vs starting from a GPT-2 model fine-tuned with SFT for 1 epoch. Compare the initial loss and final reward accuracy. Why does starting from SFT matter?

5. **Iterative DPO**: Implement a simple two-round iterative DPO: train round 1 on original preference pairs, generate new responses with the round-1 model, label them with the proxy reward function, and train round 2 on these fresh preferences. Does round 2 improve over round 1?

---

## 8. References

- [Rafailov et al. (2023) -- Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)
- [Azar et al. (2024) -- A General Theoretical Paradigm to Understand Learning from Human Feedback (IPO)](https://arxiv.org/abs/2310.12036)
- [Meng et al. (2024) -- SimPO: Simple Preference Optimization with a Reference-Free Reward](https://arxiv.org/abs/2405.14734)
- [Zhao et al. (2023) -- SLiC-HF: Sequence Likelihood Calibration with Human Feedback](https://arxiv.org/abs/2305.10425)
- [trl DPOTrainer Documentation](https://huggingface.co/docs/trl/dpo_trainer)